# E4 NF Geometry Debug

Fast, single-condition debug notebook for the E4 tiny-MLP weight-space experiment.

It does not run LR tuning, downstream optimization, or E0-E3. It only trains one RQ-spline NF on the E4 geometry objective and checks whether the held-out pullback geometry improves. Set `flow_architecture='realnvp'` only for the affine-coupling baseline.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display

from post_train_research.loss_landscape_analysis.flow_preconditioning.optimization import (
    run_direct_curve,
    run_direct_curves_batched,
    run_flow_curve,
    run_flow_curves_batched,
)

from post_train_research.loss_landscape_analysis.flow_preconditioning.e4_debug import (
    E4DebugConfig,
    build_e4_debug_state,
    train_e4_debug_flow,
    run_probe_metric_curve,
    run_probe_metric_curves,
    preconditioner_alignment_rows,
    trajectory_step_alignment_rows,
)

plt.rcParams['figure.dpi'] = 130


## Hardcoded Debug Config

Defaults are intentionally small so the notebook can be rerun while debugging. Increase `FLOW_STEPS`, `FLOW_RANDOM_SAMPLES`, or eval sample counts only after the fast run gives a clear signal.


In [ ]:
RUN_LABEL = 'e4_rq_spline_geometry_debug_seed0_rho1e2_fast'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

DEBUG_CFG = E4DebugConfig(
    run_label=RUN_LABEL,
    artifact_root='artifacts/loss_landscape_analysis/flow_preconditioning/e4_debug',
    device=DEVICE,
    dtype='float32',
    seed=0,
    rho=1e-2,
    flow_steps=150,
    eval_every=15,
    flow_batch_size=8,
    flow_lr=1e-3,
    flow_grad_clip_norm=10.0,
    flow_architecture='rq_spline',
    flow_num_layers=8,
    flow_hidden_dim=64,
    flow_network_depth=2,
    flow_log_scale_clamp=1.5,
    flow_spline_bins=8,
    flow_spline_bound=5.0,
    flow_random_samples=64,
    flow_trajectory_count=4,
    flow_trajectory_steps=8,
    heldout_geometry_samples=32,
    train_eval_samples=16,
    heldout_eval_samples=16,
)

DEBUG_CFG


## Build E4 Probe And Pools


In [ ]:
state = build_e4_debug_state(DEBUG_CFG)

print('output_dir:', state.output_dir)
print('device:', state.flow_pool.device)
print('flow_pool:', tuple(state.flow_pool.shape))
print('heldout_pool:', tuple(state.heldout_pool.shape))
print('train_eval_pool:', tuple(state.train_eval_pool.shape))
print('heldout_eval_pool:', tuple(state.heldout_eval_pool.shape))
print('probe output dim:', int(state.probe(state.flow_pool[0]).numel()))


## Train One NF And Evaluate Geometry During Training


In [ ]:
result = train_e4_debug_flow(state)
history = result.history
final_geometry = result.final_geometry

print('output_dir:', result.output_dir)
print('history rows:', len(history))
print('final geometry rows:', len(final_geometry))
display(history.tail(8))
display(final_geometry)


## Fast Sanity Checks


In [ ]:
step0 = history.iloc[0]
last = history.iloc[-1]

identity_train_abs = abs(float(step0['train_flow_R']) - float(step0['train_original_R']))
identity_heldout_abs = abs(float(step0['heldout_flow_R']) - float(step0['heldout_original_R']))
train_delta = float(last['train_R_delta'])
heldout_delta = float(last['heldout_R_delta'])
train_ratio = float(last['train_R_ratio'])
heldout_ratio = float(last['heldout_R_ratio'])

print(f"identity mismatch train_R={identity_train_abs:.6g} heldout_R={identity_heldout_abs:.6g}")
print(f"final train delta={train_delta:.6g} ratio={train_ratio:.4f}")
print(f"final heldout delta={heldout_delta:.6g} ratio={heldout_ratio:.4f}")

if identity_train_abs > 1e-3 or identity_heldout_abs > 1e-3:
    diagnosis = 'BUG: identity-initialized flow does not match original geometry. Check flow.inverse / probe_jacobians_for_flow wiring.'
elif heldout_delta < 0.0:
    diagnosis = 'OK: heldout geometry improved in this fast debug run.'
elif train_delta < 0.0 and heldout_delta >= 0.0:
    diagnosis = 'NF improves train-pool geometry but not heldout. This points to overfit / pool mismatch / objective generalization.'
else:
    diagnosis = 'NF does not even improve train-pool geometry. Debug optimizer, objective sign, flow capacity, or gradient path.'

Markdown(f"### Diagnosis\n{diagnosis}")


## Geometry Curves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

ax = axes[0, 0]
ax.plot(history['step'], history['train_flow_R'], label='train flow R')
ax.plot(history['step'], history['heldout_flow_R'], label='heldout flow R')
ax.axhline(float(history['train_original_R'].iloc[0]), linestyle='--', color='C0', alpha=0.5, label='train original R')
ax.axhline(float(history['heldout_original_R'].iloc[0]), linestyle='--', color='C1', alpha=0.5, label='heldout original R')
ax.set_xlabel('NF step')
ax.set_ylabel('global R, lower is better')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[0, 1]
plot_rows = history[history['step'] > 0]
ax.plot(plot_rows['step'], plot_rows['batch_loss'], label='train batch objective')
ax.set_xlabel('NF step')
ax.set_ylabel('batch objective')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(history['step'], history['train_flow_cond_median'], label='train flow cond median')
ax.plot(history['step'], history['heldout_flow_cond_median'], label='heldout flow cond median')
ax.set_xlabel('NF step')
ax.set_ylabel('pullback metric eig cond median')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 1]
ax.plot(history['step'], history['train_flow_flow_displacement_median'], label='train displacement')
ax.plot(history['step'], history['heldout_flow_flow_displacement_median'], label='heldout displacement')
ax.set_xlabel('NF step')
ax.set_ylabel('median |flow(theta)-theta|')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

figure_path = Path(result.output_dir) / 'geometry_debug_curves.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', figure_path)


## Small Raw vs NF vs Probe-Metric Optimization Comparison

This is a fixed-configuration sanity check, not LR tuning. It optimizes the same fresh E4 MLP starts with raw Adam, trained-NF-coordinate Adam, and a direct probe-metric step `theta <- theta - eta * (J_P(theta)^T J_P(theta) + lambda I)^(-1) grad L(theta)`.


In [ ]:
OPT_COMPARE_STARTS = 8
OPT_COMPARE_STEPS = 150
OPT_COMPARE_OPTIMIZER = 'adam'
OPT_COMPARE_LR = 3e-3
OPT_COMPARE_PROBE_LR = 1e-2
OPT_COMPARE_PROBE_DAMPING = 1e-3
OPT_COMPARE_SEED = 90_001

compare_starts = state.problem.sample_starts(OPT_COMPARE_STARTS, seed=OPT_COMPARE_SEED)
compare_lrs = torch.full(
    (OPT_COMPARE_STARTS,),
    float(OPT_COMPARE_LR),
    device=compare_starts.device,
    dtype=compare_starts.dtype,
)

raw_curves = run_direct_curves_batched(
    train_loss_fn=state.problem.train_loss,
    test_loss_fn=state.problem.test_loss,
    theta0_batch=compare_starts,
    optimizer_name=OPT_COMPARE_OPTIMIZER,
    lrs=compare_lrs,
    steps=OPT_COMPARE_STEPS,
)

nf_curves = run_flow_curves_batched(
    train_loss_fn=state.problem.train_loss,
    test_loss_fn=state.problem.test_loss,
    flow=result.flow,
    theta0_batch=compare_starts,
    optimizer_name=OPT_COMPARE_OPTIMIZER,
    lrs=compare_lrs,
    steps=OPT_COMPARE_STEPS,
)

probe_metric_curves = run_probe_metric_curves(
    train_loss_fn=state.problem.train_loss,
    test_loss_fn=state.problem.test_loss,
    probe=state.probe,
    theta0_batch=compare_starts,
    lr=OPT_COMPARE_PROBE_LR,
    steps=OPT_COMPARE_STEPS,
    damping=OPT_COMPARE_PROBE_DAMPING,
)

curve_rows = []
summary_rows = []
method_specs = [
    ('raw', raw_curves, OPT_COMPARE_OPTIMIZER, OPT_COMPARE_LR, float('nan')),
    ('nf', nf_curves, OPT_COMPARE_OPTIMIZER, OPT_COMPARE_LR, float('nan')),
    ('probe_metric', probe_metric_curves, 'probe_metric_sgd', OPT_COMPARE_PROBE_LR, OPT_COMPARE_PROBE_DAMPING),
]
for method, curves, optimizer_name, lr_value, damping_value in method_specs:
    for start_index, curve in enumerate(curves):
        for step, (train_loss, test_loss) in enumerate(zip(curve.train_loss, curve.test_loss, strict=True)):
            curve_rows.append(
                {
                    'method': method,
                    'optimizer': optimizer_name,
                    'lr': lr_value,
                    'damping': damping_value,
                    'start_index': start_index,
                    'step': step,
                    'train_loss': float(train_loss),
                    'test_loss': float(test_loss),
                }
            )
        summary_rows.append(
            {
                'method': method,
                'optimizer': optimizer_name,
                'lr': lr_value,
                'damping': damping_value,
                'start_index': start_index,
                'initial_train_loss': float(curve.train_loss[0]),
                'final_train_loss': float(curve.train_loss[-1]),
                'best_train_loss': float(curve.train_loss.min()),
                'initial_test_loss': float(curve.test_loss[0]),
                'final_test_loss': float(curve.test_loss[-1]),
                'best_test_loss': float(curve.test_loss.min()),
            }
        )

optimization_curves = pd.DataFrame(curve_rows)
optimization_summary = pd.DataFrame(summary_rows)
optimization_curves.to_csv(Path(result.output_dir) / 'small_raw_nf_probe_metric_optimization_curves.csv', index=False)
optimization_summary.to_csv(Path(result.output_dir) / 'small_raw_nf_probe_metric_optimization_summary.csv', index=False)

aggregate_summary = optimization_summary.groupby('method').agg(
    final_train_median=('final_train_loss', 'median'),
    best_train_median=('best_train_loss', 'median'),
    final_test_median=('final_test_loss', 'median'),
    best_test_median=('best_test_loss', 'median'),
).reset_index()

display(aggregate_summary)
display(optimization_summary.sort_values(['start_index', 'method']))
print('saved:', Path(result.output_dir) / 'small_raw_nf_probe_metric_optimization_curves.csv')
print('saved:', Path(result.output_dir) / 'small_raw_nf_probe_metric_optimization_summary.csv')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for ax, loss_col, title in [
    (axes[0], 'train_loss', 'train loss'),
    (axes[1], 'test_loss', 'test loss'),
]:
    for method, frame in optimization_curves.groupby('method'):
        grouped = frame.groupby('step')[loss_col]
        median = grouped.median()
        q25 = grouped.quantile(0.25)
        q75 = grouped.quantile(0.75)
        ax.plot(median.index, median.values, label=method)
        ax.fill_between(median.index, q25.values, q75.values, alpha=0.12)
    ax.set_yscale('log')
    ax.set_xlabel('optimization step')
    ax.set_ylabel(title)
    ax.set_title(f'{title}: raw theta vs trained NF vs probe metric')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

opt_figure_path = Path(result.output_dir) / 'small_raw_nf_probe_metric_optimization.png'
fig.savefig(opt_figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', opt_figure_path)


## Preconditioner Direction Diagnostics On Trajectory Points

**Plot 1 is not a self-consistency check.** It asks whether the trained NF-induced local preconditioner is close to the explicit probe-metric preconditioner at points sampled from each trajectory. `probe_metric_path` there means only “evaluate at points visited by the probe-metric optimizer”; `cos(p_NF, p_metric)` is not expected to be 1 unless NF actually learned the same metric inverse.

**Plot 2 is the self-consistency check.** It compares the actual consecutive optimizer displacement `theta_t - theta_{t+1}` against `p_metric` and `p_NF`. For `probe_metric_path`, `cos(theta_t - theta_{t+1}, p_metric)` should be close to 1.

Definitions: `p_metric = (J_P(theta)^T J_P(theta) + lambda I)^(-1) g`, `p_NF = J_inv J_inv^T g`, where `g = grad L(theta)` and `J_inv = d NF^{-1}(u) / du`.


In [ ]:
ALIGN_STARTS = min(2, OPT_COMPARE_STARTS)
ALIGN_STEPS = min(OPT_COMPARE_STEPS, 80)
ALIGN_EVERY = 10
ALIGN_PROBE_DAMPING = OPT_COMPARE_PROBE_DAMPING

alignment_rows = []
step_alignment_rows = []
for start_index in range(ALIGN_STARTS):
    theta0 = compare_starts[start_index]
    raw_path_curve = run_direct_curve(
        train_loss_fn=state.problem.train_loss,
        test_loss_fn=state.problem.test_loss,
        theta0=theta0,
        optimizer_name=OPT_COMPARE_OPTIMIZER,
        lr=OPT_COMPARE_LR,
        steps=ALIGN_STEPS,
        store_path=True,
    )
    nf_path_curve = run_flow_curve(
        train_loss_fn=state.problem.train_loss,
        test_loss_fn=state.problem.test_loss,
        flow=result.flow,
        theta0=theta0,
        optimizer_name=OPT_COMPARE_OPTIMIZER,
        lr=OPT_COMPARE_LR,
        steps=ALIGN_STEPS,
        store_path=True,
    )
    probe_path_curve = run_probe_metric_curve(
        train_loss_fn=state.problem.train_loss,
        test_loss_fn=state.problem.test_loss,
        probe=state.probe,
        theta0=theta0,
        lr=OPT_COMPARE_PROBE_LR,
        steps=ALIGN_STEPS,
        damping=ALIGN_PROBE_DAMPING,
        store_path=True,
    )

    selected_steps = list(range(0, ALIGN_STEPS + 1, ALIGN_EVERY))
    if selected_steps[-1] != ALIGN_STEPS:
        selected_steps.append(ALIGN_STEPS)
    full_steps = list(range(0, ALIGN_STEPS + 1))
    for source, curve in [
        ('raw_path', raw_path_curve),
        ('nf_path', nf_path_curve),
        ('probe_metric_path', probe_path_curve),
    ]:
        theta_points = torch.as_tensor(
            curve.path[selected_steps],
            device=state.flow_pool.device,
            dtype=state.flow_pool.dtype,
        )
        alignment_rows.extend(
            preconditioner_alignment_rows(
                train_loss_fn=state.problem.train_loss,
                probe=state.probe,
                flow=result.flow,
                theta_points=theta_points,
                damping=ALIGN_PROBE_DAMPING,
                source=source,
                start_index=start_index,
                steps=selected_steps,
            )
        )
        theta_path = torch.as_tensor(
            curve.path,
            device=state.flow_pool.device,
            dtype=state.flow_pool.dtype,
        )
        step_alignment_rows.extend(
            trajectory_step_alignment_rows(
                train_loss_fn=state.problem.train_loss,
                probe=state.probe,
                flow=result.flow,
                theta_path=theta_path,
                damping=ALIGN_PROBE_DAMPING,
                source=source,
                start_index=start_index,
                steps=full_steps,
            )
        )

alignment_df = pd.DataFrame(alignment_rows)
alignment_path = Path(result.output_dir) / 'preconditioner_alignment_diagnostic.csv'
alignment_df.to_csv(alignment_path, index=False)

step_alignment_df = pd.DataFrame(step_alignment_rows)
step_alignment_path = Path(result.output_dir) / 'trajectory_step_alignment_diagnostic.csv'
step_alignment_df.to_csv(step_alignment_path, index=False)

alignment_summary = alignment_df.groupby('source').agg(
    cos_p_nf_p_metric_median=('cos_p_nf_p_metric', 'median'),
    norm_ratio_metric_to_nf_median=('norm_ratio_metric_to_nf', 'median'),
    cos_g_p_metric_median=('cos_g_p_metric', 'median'),
    cos_g_p_nf_median=('cos_g_p_nf', 'median'),
    p_metric_norm_median=('p_metric_norm', 'median'),
    p_nf_norm_median=('p_nf_norm', 'median'),
).reset_index()
step_alignment_summary = step_alignment_df.groupby('source').agg(
    cos_step_p_metric_median=('cos_step_p_metric', 'median'),
    cos_step_p_nf_median=('cos_step_p_nf', 'median'),
    step_norm_over_p_metric_norm_median=('step_norm_over_p_metric_norm', 'median'),
    step_norm_over_p_nf_norm_median=('step_norm_over_p_nf_norm', 'median'),
).reset_index()

display(alignment_summary)
display(step_alignment_summary)
display(alignment_df.head(30))
display(step_alignment_df.head(30))
print('saved:', alignment_path)
print('saved:', step_alignment_path)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
plot_specs = [
    (axes[0, 0], 'cos_p_nf_p_metric', 'theoretical cos(p_NF, p_metric)'),
    (axes[0, 1], 'norm_ratio_metric_to_nf', 'theoretical ||p_metric|| / ||p_NF||'),
    (axes[1, 0], 'cos_g_p_metric', 'theoretical cos(g, p_metric)'),
    (axes[1, 1], 'cos_g_p_nf', 'theoretical cos(g, p_NF)'),
]
for ax, column, title in plot_specs:
    for source, frame in alignment_df.groupby('source'):
        grouped = frame.groupby('step')[column]
        median = grouped.median()
        ax.plot(median.index, median.values, marker='o', label=source)
    ax.set_xlabel('trajectory step')
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

alignment_figure_path = Path(result.output_dir) / 'preconditioner_alignment_diagnostic.png'
fig.savefig(alignment_figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', alignment_figure_path)
step_alignment_plot_df = step_alignment_df[step_alignment_df['step'] % ALIGN_EVERY == 0].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, column, title in [
    (axes[0], 'cos_step_p_metric', 'cos(theta_t - theta_{t+1}, p_metric)'),
    (axes[1], 'cos_step_p_nf', 'cos(theta_t - theta_{t+1}, p_NF)'),
]:
    for source, frame in step_alignment_plot_df.groupby('source'):
        grouped = frame.groupby('step')[column]
        median = grouped.median()
        ax.plot(median.index, median.values, marker='o', label=source)
    ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_xlabel('trajectory step')
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)

step_alignment_figure_path = Path(result.output_dir) / 'trajectory_step_alignment_diagnostic.png'
fig.savefig(step_alignment_figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', step_alignment_figure_path)


## Final Compact Table


In [ ]:
cols = [
    'step',
    'batch_loss',
    'train_original_R',
    'train_flow_R',
    'train_R_delta',
    'train_R_ratio',
    'heldout_original_R',
    'heldout_flow_R',
    'heldout_R_delta',
    'heldout_R_ratio',
    'train_flow_trace_cv',
    'heldout_flow_trace_cv',
    'train_flow_cond_median',
    'heldout_flow_cond_median',
    'train_flow_flow_cond_median',
    'heldout_flow_flow_cond_median',
]
compact = history[cols].copy()
display(compact)
compact.to_csv(Path(result.output_dir) / 'compact_history.csv', index=False)
print('saved:', Path(result.output_dir) / 'compact_history.csv')


## Files Written


In [ ]:
for path in sorted(Path(result.output_dir).iterdir()):
    print(path.name, path.stat().st_size)
